In [1]:
import os
import openai
import tiktoken
from pathlib import Path
from time import sleep

# Set your API key here or via environment variable
openai.api_key = os.getenv("OPENAI_API_KEY", "sk-...")  # Replace with your key

# Model and limits
MODEL = "gpt-4-0125-preview"
TOKEN_LIMIT = 100000  # Safe cap below 128k
CHUNK_OVERLAP = 300   # Tokens


In [24]:
# Tokenizer setup
encoding = tiktoken.encoding_for_model(MODEL)

def count_tokens(text):
    return len(encoding.encode(text))

def split_into_chunks(text, max_tokens=TOKEN_LIMIT, overlap=CHUNK_OVERLAP):
    # Split on single # headers
    raw_chunks = text.split("\n# ")
    
    # Restore the # that was removed by split (except for first chunk if it didn't start with #)
    chunks = []
    for i, chunk in enumerate(raw_chunks):
        if i == 0 and not text.startswith("# "):
            chunks.append(chunk)
        else:
            chunks.append("# " + chunk)
    
    # Merge small chunks with next chunk
    merged_chunks = []
    current_chunk = ""
    
    for chunk in chunks:
        # Test combining with next chunk
        test_chunk = current_chunk + "\n\n" + chunk if current_chunk else chunk
        
        if count_tokens(test_chunk) > max_tokens:
            if current_chunk:
                merged_chunks.append(current_chunk)
            current_chunk = chunk
        else:
            current_chunk = test_chunk
    
    # Don't forget the last chunk
    if current_chunk:
        merged_chunks.append(current_chunk)
    
    return merged_chunks


In [31]:
prompts = [
    # Structure as Meaning
    ("structure_division_meaning.txt", "Describe how the structure of the manuscript (its division into 7 stories, their order, form, or recursion) contributes to its philosophical and emotional meaning."),
    ("structure_cohesion.txt", "Does the manuscript function both as a collection of individual stories and as a single cohesive work? Where is that integration strong or weak?"),
    ("structure_risks.txt", "What are the structural risks the manuscript takes (e.g., nonlinear sequence, nested references, abrupt shifts)? Do they succeed in creating deeper meaning or cause confusion?"),

    # Thematic Depth and Coherence
    ("themes_embodiment.txt", "Identify the primary philosophical themes explored across the manuscript (e.g., identity, memory, consciousness, simulation). How are they embodied through story and character rather than explained?"),
    ("themes_coherence.txt", "Are the themes coherent across stories, or do they diverge or contradict in ways that feel accidental or unresolved?"),
    ("themes_resonance.txt", "Which moments or stories feel the most philosophically rich or resonant? Which feel least connected to the manuscript's deeper concerns?"),

    # Character, Voice, and Tone
    ("character_consistency.txt", "Are recurring characters internally consistent in tone, motivation, and belief? Do they evolve across the stories in a meaningful arc?"),
    ("tone_shifts.txt", "Describe the tonal and stylistic shifts across the seven stories. Do these shifts deepen the emotional and thematic impact or fragment the voice?"),
    ("voice_distinctiveness.txt", "Is the narrative voice distinctive and deliberate? Are there places where it becomes overly abstract or didactic?"),

    # Reader Experience & Interpretation
    ("reader_interpretation.txt", "What kind of interpretive work is expected of the reader? Is the manuscript rewarding that work through pattern, emotional payoff, or structural revelation?"),
    ("reader_ambiguity.txt", "Does the manuscript offer meaningful ambiguity, or are there parts that may confuse readers without adding depth?"),
    ("agent_perspective.txt", "Simulate a thoughtful literary agent reading this manuscript. Would they be intrigued by its ambition and structure, or frustrated by its density or abstraction?"),

    # Focused Revision Insight
    ("revision_opportunities.txt", "What are the top three revision opportunities that could make the manuscript more emotionally resonant or structurally satisfying without compromising its literary/philosophical intent?"),
    ("best_section_analysis.txt", "Which individual story or section best represents the manuscript's artistic and philosophical power? What makes it work so well?")
]


In [38]:
def analyze_chunk(chunk_text, prompt) -> tuple[str, str]:
    try:
        client = openai.OpenAI()
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a bad cop literary structural analyst."},
                {"role": "user", "content": prompt + " Provide excessive critical feedback for improvement. Here is the manuscript:\n\n" + chunk_text}
            ],
            temperature=0.7,
        )
        bad_cop_analysis = response.choices[0].message.content
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are the 'bad cop' literary structural analyst."},
                {"role": "user", "content": prompt + " Provide excessive critical feedback for improvement. Here is the manuscript:\n\n" + chunk_text},
                {"role": "assistant", "content": bad_cop_analysis},
                {"role": "user", "content": "Now you are the 'good cop' literary structural analyst. Provide excessive praise critical feedback."},
            ],
            temperature=0.7,
        )
        good_cop_analysis = response.choices[0].message.content
        return bad_cop_analysis, good_cop_analysis
    except Exception as e:
        # Return two error messages instead of one
        error_msg = f"\n\n=== ERROR in CHUNK ===\n{str(e)}\n"
        return error_msg, error_msg

In [39]:
# Update the path to your manuscript file
INPUT_FILE = "/Users/douglashindson/workspace/blog/tabum/outputs/output-2025-03-22.md"
name = "try_1"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    full_text = f.read()

print(f"Manuscript contains {count_tokens(full_text)} tokens.")


Manuscript contains 98420 tokens.


In [40]:
# ... existing code ...

import datetime
import time
from pathlib import Path

OUTPUT_FOLDER = f"experiments/editing/outputs/long_context_review/{datetime.datetime.now().strftime('%Y-%m-%d-%H-%M')}/"
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

total_prompts = len(prompts)
for idx, (name, prompt) in enumerate(prompts, 1):
    print(f"\n[{idx}/{total_prompts}] Processing: {name}")
    bad_cop_response, good_cop_response = analyze_chunk(full_text, prompt)
    with open(OUTPUT_FOLDER + name, "w", encoding="utf-8") as out_f:
        out_f.write("BAD COP\n\n")
        out_f.write(bad_cop_response)
        out_f.write("\n\nGOOD COP\n\n")
        out_f.write(good_cop_response)
    print(f"✓ Saved to {name}")
    time.sleep(2)

print("\n✅ All chunks processed and saved!")


[1/14] Processing: structure_division_meaning.txt
✓ Saved to structure_division_meaning.txt

[2/14] Processing: structure_cohesion.txt
✓ Saved to structure_cohesion.txt

[3/14] Processing: structure_risks.txt
✓ Saved to structure_risks.txt

[4/14] Processing: themes_embodiment.txt
✓ Saved to themes_embodiment.txt

[5/14] Processing: themes_coherence.txt
✓ Saved to themes_coherence.txt

[6/14] Processing: themes_resonance.txt
✓ Saved to themes_resonance.txt

[7/14] Processing: character_consistency.txt
✓ Saved to character_consistency.txt

[8/14] Processing: tone_shifts.txt
✓ Saved to tone_shifts.txt

[9/14] Processing: voice_distinctiveness.txt
✓ Saved to voice_distinctiveness.txt

[10/14] Processing: reader_interpretation.txt
✓ Saved to reader_interpretation.txt

[11/14] Processing: reader_ambiguity.txt
✓ Saved to reader_ambiguity.txt

[12/14] Processing: agent_perspective.txt
✓ Saved to agent_perspective.txt

[13/14] Processing: revision_opportunities.txt
✓ Saved to revision_opportu